Лычаный НА N33471

Вариант 3

Импортируем библиотеки.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

Маунтим Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Открываем датасет из гугл диска

In [3]:
df = pd.read_csv('/content/drive/MyDrive/itmo/3/теория алгоритмов/lab_5/Chicago_hotels.csv', sep=';')
df.head()

,date1,date2,x1,x2,x3,x4
0,January,1994,"44,3","104,04","51,9","72,15"
1,February,1994,"54,7","102,74","60,1","73,47"
2,March,1994,"61,9","105,23",65,"74,91"
3,April,1994,"69,1","113,63","69,8","79,06"
4,May,1994,"70,8","120,77","72,4","82,07"


Размаунтим Google Drive

In [4]:
drive.flush_and_unmount()

Конвертируем x3 to float

In [5]:
df['x3'] = df['x3'].apply(lambda x: x.replace(',','.'))
df['x3'] = df['x3'].transform(pd.to_numeric, errors='coerce')

Создаем столбец с датой и номером месяца

In [6]:
df['date'] = df['date1'].values + df['date2'].apply(lambda x: ' ' + str(x)).values
df['date'] = pd.to_datetime(df['date'])
month_to_number = dict(zip(df.date1.unique(), range(1, len(df['date1'])+1)))
df['month'] = df['date1'].apply(lambda x: month_to_number[x])

График загрузки гостиниц в процентах от времени

In [7]:
fig = px.line(df, x='date', y="x3")
fig.show()

Выборка для обучающей модели

In [8]:
df_train = df[:-8]

Создадим ряд логарифмов для исходных данных

In [9]:
df_train['logx3'] = np.log10(df_train['x3'].values)

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



График логарифма загрузки гостиниц в процентах от времени

In [10]:
fig = px.line(df_train, x='date', y="x3")
fig.show()

In [11]:
X_train, Y_train = df_train[['month','date2']], df_train['x3']

In [12]:
X_test = df[-8:][['month','date2']]

Модель LinearRegression, KNeighborsRegressor, RandomForestRegressor

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

In [14]:
prediction_LR = LinearRegression().fit(X_train, Y_train).predict(X_test)
prediction_KNR = KNeighborsRegressor().fit(X_train, Y_train).predict(X_test)
prediction_RFR= RandomForestRegressor().fit(X_train, Y_train).predict(X_test)

Добавляем полученные данные к датасету

In [15]:
new_x3_LR = np.concatenate((df_train.x3.values, prediction_LR))
df['x3_LR'] = new_x3_LR
new_x3_KNR = np.concatenate((df_train.x3.values, prediction_KNR))
df['x3_KNR'] = new_x3_KNR
new_x3_RFR = np.concatenate((df_train.x3.values, prediction_RFR))
df['x3_RFR'] = new_x3_RFR

График загрузки гостиниц в процентах от времени c предсказаниями

In [16]:
fig = px.line(df, x = 'date', y = 'x3')
fig.add_scatter(x=df['date'], y=df['x3_LR'], mode = 'lines', name='LR')
fig.add_scatter(x=df['date'], y=df['x3_KNR'], mode = 'lines', name='KNR')
fig.add_scatter(x=df['date'], y=df['x3_RFR'], mode = 'lines', name='RFR')
fig.show()

Вывод: лучший результат показывает алгоритм RandomForestRegressor с учетом сезонности модели